In [2]:
from google.colab import drive
drive.mount('/content/drive')
import cv2
import numpy as np
import json
import os
import csv

from PIL import Image

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def order_points(pts):
    rect = np.zeros((4, 2), dtype="float32")

    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]

    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]

    return rect


def four_point_transform(image, pts):
    rect = order_points(pts)
    (tl, tr, br, bl) = rect

    widthA = np.linalg.norm(br - bl)
    widthB = np.linalg.norm(tr - tl)
    maxWidth = max(int(widthA), int(widthB))

    heightA = np.linalg.norm(tr - br)
    heightB = np.linalg.norm(tl - bl)
    maxHeight = max(int(heightA), int(heightB))

    dst = np.array([
        [0, 0],
        [maxWidth - 1, 0],
        [maxWidth - 1, maxHeight - 1],
        [0, maxHeight - 1]], dtype="float32")

    M = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(image, M, (maxWidth, maxHeight))

    return warped


image = cv2.imread("drive/MyDrive/MyIndividualProject/raw_images/full_pdf.pdf")
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (5, 5), 0)

edged = cv2.Canny(blur, 75, 200)

contours, _ = cv2.findContours(edged, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
contours = sorted(contours, key=cv2.contourArea, reverse=True)

for c in contours:
    peri = cv2.arcLength(c, True)
    approx = cv2.approxPolyDP(c, 0.02 * peri, True)

    if len(approx) == 4:
        screenCnt = approx
        break

warped = four_point_transform(image, screenCnt.reshape(4, 2))

cv2.imwrite("drive/MyDrive/MyIndividualProject/aligned.pdf", warped)

error: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cv::cvtColor'


In [ ]:
with open("drive/MyDrive/MyIndividualProject/templates/template.json", "r", encoding="utf-8") as f:
    template = json.load(f)

image = cv2.imread("aligned.jpg")

os.makedirs("drive/MyDrive/MyIndividualProject/dataset/cells", exist_ok=True)

for question_id, box in template["answers"].items():
    x1, y1, x2, y2 = box

    crop = image[y1:y2, x1:x2]

    filename = f"dataset/cells/{question_id}.png"
    cv2.imwrite(filename, crop)

    print("saved", filename)

In [ ]:
RAW_DIR = "drive/MyDrive/MyIndividualProject/raw_images"
OUTPUT_DIR = "drive/MyDrive/MyIndividualProject/dataset/cells"

os.makedirs(OUTPUT_DIR, exist_ok=True)

with open("template.json", "r", encoding="utf-8") as f:
    template = json.load(f)

rows = []

for image_name in os.listdir(RAW_DIR):
    image_path = os.path.join(RAW_DIR, image_name)

    image = cv2.imread(image_path)

    for question_id, box in template["answers"].items():
        x1, y1, x2, y2 = box

        crop = image[y1:y2, x1:x2]

        output_name = f"{image_name}_{question_id}.png"
        output_path = os.path.join(OUTPUT_DIR, output_name)

        cv2.imwrite(output_path, crop)

        rows.append([
            output_name,
            ""
        ])

with open("dataset/labels.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["image", "label"])
    writer.writerows(rows)

In [ ]:
transform = transforms.Compose([
    transforms.RandomRotation(5),
    transforms.RandomAffine(0, shear=5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
])

input_dir = "drive/MyDrive/MyIndividualProject/dataset/cells"
output_dir = "drive/MyDrive/MyIndividualProject/dataset/augmented"

os.makedirs(output_dir, exist_ok=True)

for filename in os.listdir(input_dir):
    path = os.path.join(input_dir, filename)

    image = Image.open(path)

    for i in range(5):
        aug = transform(image)
        aug.save(os.path.join(output_dir, f"{i}_{filename}"))